In [1]:
import os,sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append('/root/liubo/TravDiT')  # 添加项目根目录到 Python 路径
from args import make_args
from datasets import Dataset
from util import load_raw_data,load_vec,generate_trajectory_prompt_temp,temporal_pattern_one_hot_batch
import random
from datetime import timedelta
import pandas as pd
import torch

args = make_args()
unique_poi_types = args.unique_poi_types
random.seed(args.seed)  # 设置随机种子以确保可重复性

In [2]:
all_texts, all_labels = [], []
dow_map = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
city_list = ['Changsha', 'Guangzhou', 'Shenzhen']
sample_size = 8000 # for one city

for city_id, city in enumerate(city_list):
    raw_data = load_raw_data(city=city)
    vec = load_vec(city=city)

    indices = random.sample(range(len(raw_data)), sample_size)
    sampled_raw_data = [raw_data[i] for i in indices]

    texts, labels = [], []
    for item in sampled_raw_data:
        traj = item["traj_region_id"]
        start_time = pd.to_datetime(item['tod'])  # 起始时间点
        i = 0
        while i < len(traj) - 1:
            start_region_id = traj[i]
            # === 计算在当前起点的停留时长 ===
            stay_slots = 1
            j = i + 1
            while j < len(traj) and traj[j] == start_region_id:
                stay_slots += 1
                j += 1
            stay_duration_hours = stay_slots * 0.5
            # 出发时间 = 离开该点的时刻
            departure_time = start_time + timedelta(minutes=30 * (i + stay_slots - 1))
            tod = departure_time.strftime("%H:%M")
            dow = dow_map[departure_time.dayofweek]
            # prompt 构造
            departure_poi_dist = vec[int(start_region_id)][2: 2 + args.poi_dim]
            prompt = generate_trajectory_prompt_temp(
                city=city,
                tod=tod,
                dow=dow,
                departure_poi_dist=departure_poi_dist,
                poi_category_list=args.unique_poi_types
            )
            texts.append(prompt)
            # 目的地 POI（第一个不同于起点的位置）
            if j < len(traj):
                dest_region_id = traj[j]
                dest_poi_dist = vec[int(dest_region_id)][2: 2 + args.poi_dim]
            else:
                dest_poi_dist = torch.zeros(args.poi_dim)
            label = [city_id] + [stay_duration_hours] + dest_poi_dist.tolist()
            labels.append(label)
            i = j  # 下一个起点

    all_texts.extend(texts)
    all_labels.extend(labels)

# ========== 构造混合 Dataset ==========
dataset = Dataset.from_dict({
    "text": all_texts,
    "label": all_labels
})

# ========== 打乱并划分 ==========
dataset = dataset.shuffle(seed=args.seed)
split_idx = int(0.8 * len(dataset))
train_dataset = dataset.select(range(split_idx))
valid_dataset = dataset.select(range(split_idx, len(dataset)))

/root/liubo/TravDiT/util.py:127: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  vec = torch.tensor(vec, dtype=torch.float32)


[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])
[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])
[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])


In [3]:
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoModel,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
)
from trainer.customTrainer import CustomTrainer
base_model  = "/datadisk/llama2"
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True,cache_dir = "./models/")
tokenizer.pad_token = tokenizer.eos_token
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512)

traj_train_df = train_dataset.map(preprocess_function, batched=True)
traj_valid_df = valid_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/68608 [00:00<?, ? examples/s]

Map:   0%|          | 0/17153 [00:00<?, ? examples/s]

In [4]:
from model.model_LLM import temporalTripModel
model = temporalTripModel(base_model_path="/datadisk/llama2")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# # Configure training parameters
training_params = TrainingArguments(
    output_dir="/checkpt",
    num_train_epochs=args.llm_epoch,
    per_device_train_batch_size=args.llm_batch_size,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_strategy='no',
    save_total_limit=0,
    load_best_model_at_end=False,
    evaluation_strategy='epoch',
    logging_steps=32,
    learning_rate=5e-5,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine"
)


# Initialize and configure the trainer
trainer = CustomTrainer(
    model= model,
    train_dataset= traj_train_df,
    eval_dataset= traj_valid_df,
    tokenizer= tokenizer,
    args= training_params,
    compute_metrics= None,
)


/root/liubo/TravDiT/.conda/lib/python3.11/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_21185/3759004214.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


In [6]:
trainer.train()

loss_poi: 0.6576,loss_duration:0.1560, total_loss: 0.8136
Accuracy: 0.0
loss_poi: 0.7186,loss_duration:0.1590, total_loss: 0.8776
Accuracy: 0.0
loss_poi: 0.6083,loss_duration:0.1626, total_loss: 0.7709
Accuracy: 0.0
loss_poi: 0.6957,loss_duration:0.1675, total_loss: 0.8632
Accuracy: 0.0


Epoch,Training Loss,Validation Loss


loss_poi: 0.4294,loss_duration:0.1626, total_loss: 0.5920
Accuracy: 0.0
loss_poi: 0.6099,loss_duration:0.1667, total_loss: 0.7766
Accuracy: 0.0
loss_poi: 0.6254,loss_duration:0.1595, total_loss: 0.7849
Accuracy: 0.0
loss_poi: 0.8790,loss_duration:0.1599, total_loss: 1.0389
Accuracy: 0.0
loss_poi: 0.7058,loss_duration:0.1634, total_loss: 0.8692
Accuracy: 0.0
loss_poi: 0.7355,loss_duration:0.1588, total_loss: 0.8943
Accuracy: 0.0
loss_poi: 0.6984,loss_duration:0.1612, total_loss: 0.8596
Accuracy: 0.0
loss_poi: 0.6464,loss_duration:0.1637, total_loss: 0.8101
Accuracy: 0.0
loss_poi: 0.6816,loss_duration:0.1607, total_loss: 0.8423
Accuracy: 0.0625
loss_poi: 0.9712,loss_duration:0.1631, total_loss: 1.1343
Accuracy: 0.0
loss_poi: 0.6865,loss_duration:0.1645, total_loss: 0.8509
Accuracy: 0.0
loss_poi: 0.5871,loss_duration:0.1659, total_loss: 0.7530
Accuracy: 0.0
loss_poi: 0.6201,loss_duration:0.1606, total_loss: 0.7807
Accuracy: 0.0
loss_poi: 0.7183,loss_duration:0.1584, total_loss: 0.8767
Acc

KeyboardInterrupt: 

In [ ]:
# Save the fine-tuned model and tokenizer
trainer.model.save_pretrained(save_path=f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm")
trainer.tokenizer.save_pretrained(save_directory=f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm")
torch.save(model.trajectory_head.state_dict(), f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/llm/trajectory_head.pt")

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
